In [1]:
"""
Three two-level atoms in a THERMAL reservoir: 2N = 6 collapse operators,
plus the angular fluorescence intensity I(theta, phi).

Companion to Sections 7.4 and 10 of "Collective Basis Sets for Three Two-Level Atoms".

Master equation (Section 7.4):

  d(rho)/dt = -i[H, rho]
      + (nbar+1) sum_mn Gamma_mn ( s_n rho s_m^+ - 1/2 { s_m^+ s_n , rho } )
      +  nbar    sum_mn Gamma_mn ( s_n^+ rho s_m^- - 1/2 { s_m^- s_n^+ , rho } )

The SAME matrix Gamma appears in both terms, so one diagonalization serves both and
the collapse operators are simply

      c_mu^down = sqrt( (nbar+1) gamma_mu ) S_mu
      c_mu^up   = sqrt(  nbar    gamma_mu ) S_mu^dagger        S_mu = sum_n conj(v^mu_n) s_n

2N = 6 operators for N = 3, NOT the 26 that a full secular (Davies) treatment produces.
The Davies construction resolves the bath separately at each of the 13 distinct Bohr
frequencies of the interacting H; since those are spread over a range ~|J_mn| while all
sitting at omega_0, and Gamma(omega) ~ omega^3 and nbar(omega) are flat to ~1e-5 across
that span, they may all be evaluated at omega_0 -- whereupon the 26 re-sum into these 6.
Structurally: the Kossakowski matrix in the basis {s_n} u {s_n^+} is block diagonal,
(nbar+1)*Gamma  +  nbar*Gamma, so its rank can never exceed 2N.

Validation status: the algebra here was checked against an independent numpy
implementation -- Liouvillian equality to 1e-15 for all geometries and several nbar,
steady state equal to the free-atom thermal product state, and the angular sum rule
int I dOmega = sum_mn Gamma_mn <s_m^+ s_n>.  Written against the QuTiP 4/5 common API.

Usage:
    python3 thermal_trimer_qutip.py
"""

import numpy as np
import qutip as qt

HBAR = 1.054571817e-34
KB = 1.380649e-23

# QuTiP: sigmam() = [[0,0],[1,0]] lowers basis(2,0) -> basis(2,1),
# so basis(2,0) is the EXCITED state.  Assert rather than trust.
assert abs((qt.sigmam() * qt.basis(2, 0)).norm() - 1.0) < 1e-12
EXCITED, GROUND = qt.basis(2, 0), qt.basis(2, 1)


# ======================================================================
# 1.  Geometry, bath occupation, operators
# ======================================================================
def lehmberg(pos, dhat, k0=1.0, Gamma=1.0):
    """Collective decay matrix Gamma_mn and coherent coupling J_mn."""
    pos = np.asarray(pos, float)
    dhat = np.asarray(dhat, float)
    dhat = dhat / np.linalg.norm(dhat)
    N = len(pos)
    Gam, Jm = np.zeros((N, N)), np.zeros((N, N))
    for m in range(N):
        Gam[m, m] = Gamma
        for n in range(N):
            if m == n:
                continue
            rv = pos[m] - pos[n]
            r = np.linalg.norm(rv)
            xi, c = k0 * r, float(dhat @ (rv / r))
            Gam[m, n] = 1.5 * Gamma * (
                (1 - c**2) * np.sin(xi) / xi
                + (1 - 3 * c**2) * (np.cos(xi) / xi**2 - np.sin(xi) / xi**3))
            Jm[m, n] = 0.75 * Gamma * (
                -(1 - c**2) * np.cos(xi) / xi
                + (1 - 3 * c**2) * (np.sin(xi) / xi**2 + np.cos(xi) / xi**3))
    return Gam, Jm


def nbar_from_temperature(nu_Hz, T_K):
    """Planck occupation nbar = 1/(exp(h nu / kT) - 1)."""
    x = 2 * np.pi * HBAR * nu_Hz / (KB * T_K)
    return 1.0 / np.expm1(x)


def atom_ops(N=3):
    return [qt.tensor([qt.sigmam() if k == n else qt.qeye(2) for k in range(N)])
            for n in range(N)]


def hamiltonian(Jm, sm):
    N = len(sm)
    H = 0 * sm[0].dag() * sm[0]
    for m in range(N):
        for n in range(N):
            if m != n:
                H += Jm[m, n] * sm[m].dag() * sm[n]
    return H


# ======================================================================
# 2.  THE SIX COLLAPSE OPERATORS
# ======================================================================
def collapse_ops_thermal(Gam, sm, nbar, tol=1e-12):
    """
    Returns (c_ops, gamma, V, S).

    c_ops : 2N operators, emission first then absorption
              c[mu]     = sqrt((nbar+1) gamma_mu) S_mu
              c[N + mu] = sqrt( nbar     gamma_mu) S_mu^dag
    gamma : mode rates (eigenvalues of Gam)
    V     : V[:, mu] = mode amplitudes v^mu_n
    S     : the bare mode operators S_mu (no rate prefactor), useful for
            mode-resolved populations <S_mu^dag S_mu>

    Modes with gamma <= tol are dropped; at nbar = 0 the absorption operators
    vanish identically and are dropped too, recovering the N-operator vacuum case.

    NOTE.  c^up = S_mu^dag is correct because Gamma is REAL SYMMETRIC (identical
    atoms, real dipole matrix elements).  The general form is
    sum_n conj(v^mu_n) s_n^+, which differs from S_mu^dag when v is complex.
    """
    Gam = np.asarray(Gam)
    gamma, V = np.linalg.eigh(Gam)
    if gamma.min() < -1e-10:
        raise ValueError(f"Gamma not positive semidefinite (min {gamma.min():.2e})")
    gamma = np.clip(gamma, 0.0, None)
    N = len(sm)
    S = [sum(np.conj(V[n, mu]) * sm[n] for n in range(N)) for mu in range(N)]
    c_ops = [np.sqrt((nbar + 1) * gamma[mu]) * S[mu]
             for mu in range(N) if gamma[mu] > tol]
    if nbar > tol:
        c_ops += [np.sqrt(nbar * gamma[mu]) * S[mu].dag()
                  for mu in range(N) if gamma[mu] > tol]
    return c_ops, gamma, V, S


def verify_thermal(Gam, Jm, sm, nbar, verbose=True):
    """Compare the Liouvillian from c_ops with the master equation written out."""
    N = len(sm)
    H = hamiltonian(Jm, sm)
    c_ops, gamma, V, S = collapse_ops_thermal(Gam, sm, nbar)

    L_direct = qt.liouvillian(H)
    for m in range(N):
        for n in range(N):
            L_direct += (nbar + 1) * Gam[m, n] * qt.lindblad_dissipator(sm[n], sm[m])
            L_direct += nbar * Gam[m, n] * qt.lindblad_dissipator(sm[n].dag(),
                                                                 sm[m].dag())
    dev = np.abs((qt.liouvillian(H, c_ops) - L_direct).full()).max()
    if verbose:
        print(f"      nbar = {nbar:<8.4g} c_ops = {len(c_ops)}   "
              f"rates(down) = {np.round((nbar + 1) * gamma, 5)}   "
              f"max|dL| = {dev:.2e}")
    assert dev < 1e-9, "collapse operators do not reproduce the master equation"
    return c_ops, gamma, V, S


# ======================================================================
# 3.  ANGULAR INTENSITY  I(theta, phi)
# ======================================================================
def unit_vector(theta, phi):
    return np.array([np.sin(theta) * np.cos(phi),
                     np.sin(theta) * np.sin(phi),
                     np.cos(theta)])


def detection_op(theta, phi, pos, sm, k0=1.0):
    """D(n) = sum_j exp(-i k0 n.r_j) s_j   (Section 10.1)."""
    nh = unit_vector(theta, phi)
    pos = np.asarray(pos, float)
    return sum(np.exp(-1j * k0 * float(nh @ pos[j])) * sm[j]
               for j in range(len(sm)))


def intensity(theta, phi, state, pos, sm, dhat, k0=1.0, Gamma=1.0,
              dipole_pattern=True):
    """
    Fluorescence power per unit solid angle in the direction (theta, phi):

        I(n) = W(n) <D^dag(n) D(n)>,   W(n) = (3 Gamma / 8 pi) [1 - (n.dhat)^2]

    Set dipole_pattern=False to get the bare interference factor <D^dag D> alone.

    THERMAL CAVEAT.  This is the normally-ordered SOURCE term -- the light radiated
    by the atomic dipoles, i.e. the fluorescence measured above background.  With a
    thermal field the free-field part of E^(+) no longer annihilates the state and
    contributes an additional (isotropic) blackbody background proportional to nbar,
    which is not included here.  Experimentally that background is what one subtracts.
    """
    D = detection_op(theta, phi, pos, sm, k0)
    val = float(np.real(qt.expect(D.dag() * D, state)))
    if dipole_pattern:
        nh = unit_vector(theta, phi)
        dh = np.asarray(dhat, float)
        dh = dh / np.linalg.norm(dh)
        val *= 3 * Gamma / (8 * np.pi) * (1 - float(nh @ dh)**2)
    return val


def intensity_map(state, pos, sm, dhat, k0=1.0, Gamma=1.0, ntheta=61, nphi=121):
    """I on a (theta, phi) grid.  Returns (theta, phi, I) with I of shape (ntheta, nphi)."""
    th = np.linspace(0, np.pi, ntheta)
    ph = np.linspace(0, 2 * np.pi, nphi)
    I = np.array([[intensity(t, p, state, pos, sm, dhat, k0, Gamma)
                   for p in ph] for t in th])
    return th, ph, I


def total_power(state, Gam, sm):
    """
    sum_mn Gamma_mn <s_m^+ s_n>  -- the angular integral of I(n).
    Checking intensity_map against this is the cheapest test that the
    detection operator and the decay matrix are mutually consistent.
    """
    N = len(sm)
    return float(np.real(sum(Gam[m, n] * qt.expect(sm[m].dag() * sm[n], state)
                             for m in range(N) for n in range(N))))


def integrate_map(th, ph, I):
    """
    int I sin(theta) dtheta dphi by the trapezoidal rule.

    Second-order accurate, so any residual against total_power() is quadrature
    error and falls by 4x per grid doubling -- verified 1.4e-4 / 3.4e-5 / 8.5e-6 /
    2.1e-6 at 91, 181, 361, 721 points for the chain.  Widely separated atoms need
    finer grids because the interference pattern oscillates faster.
    """
    return np.trapezoid(np.trapezoid(I * np.sin(th)[:, None], ph, axis=1), th) \
        if hasattr(np, "trapezoid") else \
        np.trapz(np.trapz(I * np.sin(th)[:, None], ph, axis=1), th)


# ======================================================================
# 4.  Geometries
# ======================================================================
def equilateral(R=0.45):
    """Dipoles perpendicular to the plane: the full S3 case of Section 3."""
    return ([[R * np.cos(2 * np.pi * n / 3), R * np.sin(2 * np.pi * n / 3), 0.0]
             for n in range(3)], [0.0, 0.0, 1.0])


def chain(d=1.2):
    """Atom 2 in the middle; dipoles transverse.  k0*d ~ 1 for the Section 4.4 regime."""
    return [[0, 0, -d], [0, 0, 0.0], [0, 0, d]], [1.0, 0.0, 0.0]


def isosceles(base=0.35, height=0.55):
    return [[-base, 0, 0], [0, height, 0], [base, 0, 0]], [0.0, 0.0, 1.0]


# ======================================================================
# 5.  Demonstrations
# ======================================================================
def demo_verification():
    print("=" * 74)
    print("1.  SIX COLLAPSE OPERATORS REPRODUCE THE THERMAL MASTER EQUATION")
    print("=" * 74)
    for label, geom in [("equilateral", equilateral()),
                        ("chain", chain()),
                        ("isosceles", isosceles())]:
        pos, dhat = geom
        sm = atom_ops(3)
        Gam, Jm = lehmberg(pos, dhat)
        print(f"  {label}")
        for nb in [0.0, 0.05, 0.5, 3.0]:
            verify_thermal(Gam, Jm, sm, nb)


def demo_steady_state():
    print("\n" + "=" * 74)
    print("2.  THERMAL STEADY STATE IS THE FREE-ATOM PRODUCT STATE")
    print("    -> no atomic coherences -> the fluorescence is ISOTROPIC")
    print("=" * 74)
    pos, dhat = equilateral()
    sm = atom_ops(3)
    Gam, Jm = lehmberg(pos, dhat)
    H = hamiltonian(Jm, sm)
    print("    nbar     p_e (per atom)   nbar/(2nbar+1)   max|coherence|   "
          "I(0)/I(pi/2)")
    for nb in [0.05, 0.5, 3.0]:
        c_ops, gamma, V, S = collapse_ops_thermal(Gam, sm, nb)
        rss = qt.steadystate(H, c_ops)
        pe = np.mean([qt.expect(sm[n].dag() * sm[n], rss) for n in range(3)])
        coh = max(abs(qt.expect(sm[m].dag() * sm[n], rss))
                  for m in range(3) for n in range(3) if m != n)
        I0 = intensity(0.0, 0.0, rss, pos, sm, dhat, dipole_pattern=False)
        I9 = intensity(np.pi / 2, 0.0, rss, pos, sm, dhat, dipole_pattern=False)
        print(f"    {nb:<8.3g} {pe:<16.8f} {nb / (2 * nb + 1):<16.8f} "
              f"{coh:<16.2e} {I0 / I9:.8f}")
    print("    The last two columns are the point: in equilibrium the atomic")
    print("    coherences vanish, so every direction radiates equally.  All of the")
    print("    angular structure in this note is a NON-equilibrium phenomenon.")


def demo_relaxation():
    print("\n" + "=" * 74)
    print("3.  ANISOTROPY DECAYS AS THE TRIMER THERMALIZES  (equilateral, |eee> start)")
    print("    theta measured from the C3 axis;  I normalised to its value at pi/2")
    print("=" * 74)
    pos, dhat = equilateral()
    sm = atom_ops(3)
    Gam, Jm = lehmberg(pos, dhat)
    H = hamiltonian(Jm, sm)
    nb = 0.3
    c_ops, gamma, V, S = collapse_ops_thermal(Gam, sm, nb)
    psi0 = qt.tensor([EXCITED] * 3)
    tlist = np.linspace(0, 25, 26)
    states = qt.mesolve(H, psi0, tlist, c_ops).states
    print("      t      I(0)      I(45deg)   I(90deg)    I(0)/I(90)   <S0+S0>  <S1+S1>")
    for i in [0, 1, 2, 4, 8, 15, 25]:
        r = states[i]
        Is = [intensity(t, 0.0, r, pos, sm, dhat, dipole_pattern=False)
              for t in (0.0, np.pi / 4, np.pi / 2)]
        m0 = np.real(qt.expect(S[2].dag() * S[2], r))   # bright mode (largest gamma)
        m1 = np.real(qt.expect(S[0].dag() * S[0], r))   # a subradiant mode
        print(f"    {tlist[i]:5.1f} {Is[0]:9.4f} {Is[1]:10.4f} {Is[2]:10.4f} "
              f"{Is[0] / Is[2]:12.4f} {m0:9.4f} {m1:8.4f}")


def demo_selection_rule():
    print("\n" + "=" * 74)
    print("4.  THE ON-AXIS SELECTION RULE IS STATE-INDEPENDENT (Section 10.4)")
    print("=" * 74)
    pos, dhat = equilateral()
    sm = atom_ops(3)
    Gam, Jm = lehmberg(pos, dhat)
    _, gamma, V, S = collapse_ops_thermal(Gam, sm, 0.3)
    # D(axis) must reduce to a multiple of the bright mode alone
    D = detection_op(0.0, 0.0, pos, sm)
    A = [sum(np.exp(-1j * float(unit_vector(0, 0) @ np.asarray(pos, float)[j]))
             * V[j, mu] for j in range(3)) for mu in range(3)]
    print(f"    |A_mu| on the C3 axis:  " +
          "   ".join(f"mode{mu}(g={gamma[mu]:.3f}): {abs(A[mu]):.3e}"
                     for mu in range(3)))
    resid = np.abs((D - sum(A[mu] * S[mu] for mu in range(3))).full()).max()
    print(f"    D(axis) - sum_mu A_mu S_mu  ->  {resid:.2e}   (operator identity)")
    print("    Since the two subradiant A_mu vanish identically, an on-axis detector")
    print("    sees only the bright mode -- at any temperature, in any state.")


def demo_sum_rule():
    print("\n" + "=" * 74)
    print("5.  SUM RULE:  int I(n) dOmega  =  sum_mn Gamma_mn <s_m^+ s_n>")
    print("=" * 74)
    for label, geom in [("equilateral", equilateral()), ("chain", chain())]:
        pos, dhat = geom
        sm = atom_ops(3)
        Gam, Jm = lehmberg(pos, dhat)
        H = hamiltonian(Jm, sm)
        for nb in [0.0, 0.4]:
            c_ops, *_ = collapse_ops_thermal(Gam, sm, nb)
            psi0 = qt.tensor([EXCITED] * 3)
            r = qt.mesolve(H, psi0, [0.0, 0.8], c_ops).states[-1]
            th, ph, I = intensity_map(r, pos, sm, dhat, ntheta=181, nphi=181)
            lhs, rhs = integrate_map(th, ph, I), total_power(r, Gam, sm)
            print(f"    {label:<12} nbar={nb:<5.2g}  integral = {lhs:.6f}   "
                  f"sum_mn = {rhs:.6f}   rel.err = {abs(lhs - rhs) / rhs:.2e}")


if __name__ == "__main__":
    demo_verification()
    demo_steady_state()
    demo_relaxation()
    demo_selection_rule()
    demo_sum_rule()


1.  SIX COLLAPSE OPERATORS REPRODUCE THE THERMAL MASTER EQUATION
  equilateral
      nbar = 0        c_ops = 3   rates(down) = [0.1176  0.1176  2.76479]   max|dL| = 0.00e+00
      nbar = 0.05     c_ops = 6   rates(down) = [0.12348 0.12348 2.90303]   max|dL| = 0.00e+00
      nbar = 0.5      c_ops = 6   rates(down) = [0.17641 0.17641 4.14719]   max|dL| = 0.00e+00
      nbar = 3        c_ops = 6   rates(down) = [ 0.47042  0.47042 11.05916]   max|dL| = 0.00e+00
  chain
      nbar = 0        c_ops = 3   rates(down) = [0.03822 0.84316 2.11863]   max|dL| = 0.00e+00
      nbar = 0.05     c_ops = 6   rates(down) = [0.04013 0.88532 2.22456]   max|dL| = 0.00e+00
      nbar = 0.5      c_ops = 6   rates(down) = [0.05732 1.26474 3.17794]   max|dL| = 0.00e+00
      nbar = 3        c_ops = 6   rates(down) = [0.15286 3.37263 8.47451]   max|dL| = 0.00e+00
  isosceles
      nbar = 0        c_ops = 3   rates(down) = [0.07895 0.09546 2.82559]   max|dL| = 0.00e+00
      nbar = 0.05     c_ops = 6   rates(dow